# Exploratory Data Analysis
Global Cybersecurity Threats (2015-2024) — Fast vs. Slow Resolution project

First working version, written without the Colab/reference material yet.
Will be revised with attribution comments once those are provided.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW_PATH = Path("../data/raw/cybersecurity_threats.csv")
FIGURES_DIR = Path("../outputs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

## 1. Load and inspect the raw data

In [ ]:
df = pd.read_csv(RAW_PATH)
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", "_", regex=True)
)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 2. Class balance check — will Fast/Slow be roughly balanced?

In [ ]:
median_time = df["incident_resolution_time_in_hours"].median()
print(f"Median resolution time: {median_time:.2f} hours")

df["resolution_class"] = (
    df["incident_resolution_time_in_hours"] >= median_time
).map({True: "Slow", False: "Fast"})

df["resolution_class"].value_counts()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="resolution_class")
plt.title("Fast vs. Slow Resolution Class Balance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_balance.png")
plt.show()

## 3. Average resolution time by defense mechanism
This is the core question for Option C — which defense mechanisms are
associated with faster containment?

In [ ]:
avg_by_defense = (
    df.groupby("defense_mechanism_used")["incident_resolution_time_in_hours"]
    .mean()
    .sort_values()
)
avg_by_defense

In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(x=avg_by_defense.values, y=avg_by_defense.index, orient="h")
plt.xlabel("Average Resolution Time (hours)")
plt.title("Average Resolution Time by Defense Mechanism")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "avg_resolution_by_defense.png")
plt.show()

## 4. Fast/Slow ratio by defense mechanism

In [ ]:
ratio_table = pd.crosstab(
    df["defense_mechanism_used"], df["resolution_class"], normalize="index"
)
ratio_table

In [ ]:
ratio_table.plot(kind="barh", stacked=True, figsize=(7, 5), colormap="coolwarm")
plt.xlabel("Proportion")
plt.title("Fast vs. Slow Proportion by Defense Mechanism")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fast_slow_ratio_by_defense.png")
plt.show()

## 5. Defense mechanism vs. attack type
Does the 'best' defense mechanism depend on the type of attack?

In [ ]:
crosstab = pd.crosstab(df["attack_type"], df["defense_mechanism_used"])

plt.figure(figsize=(9, 6))
sns.heatmap(crosstab, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Attack Type vs. Defense Mechanism (counts)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "attack_type_vs_defense.png")
plt.show()

## 6. Notes / takeaways
_(fill in after running — e.g. which defense mechanism had the lowest
average resolution time, whether the effect holds across attack types,
anything surprising in the class balance)_